In [45]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

In [ ]:
merged_df = pd.read_csv('C:\Users\Admin\Documents\Uni\DS108\Data\Silver Data\Silver_Data.csv')

## Chuyển sang Gold Data

In [47]:
# Xóa những cột không cần thiết
columns_to_drop = ['gia_truoc']
merged_df.drop(columns=columns_to_drop, inplace=True)

## Tính giá trị thực cho các khách hàng mới

In [48]:
# Bỏ trùng tên khách sạn
hotel_df = merged_df.drop_duplicates(subset='ten', keep='first')

# Các tham số cho công thức Netflix
R = hotel_df['diem_danh_gia']
v = hotel_df['so_luong_danh_gia']
C = hotel_df['diem_danh_gia'].mean()
m = hotel_df['so_luong_danh_gia'].quantile(0.75)

# Tính điểm theo Bayesian Rating (Netflix-style)
hotel_df['gia_tri_thuc_netflix'] = ((v / (v + m)) * R + (m / (v + m)) * C).round(2)

# Gán lại giá trị này về merged_df thông qua cột 'ten'
merged_df = merged_df.merge(
    hotel_df[['ten', 'gia_tri_thuc_netflix']],
    on='ten',
    how='left',
    suffixes=('', '_new')
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_2540\1015755860.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hotel_df['gia_tri_thuc_netflix'] = ((v / (v + m)) * R + (m / (v + m)) * C).round(2)


In [49]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16270 entries, 0 to 16269
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ten                   16270 non-null  object 
 1   dia_chi               16270 non-null  object 
 2   diem_danh_gia         16270 non-null  float64
 3   so_luong_danh_gia     16270 non-null  float64
 4   vi_tri                16270 non-null  float64
 5   dich_vu               16270 non-null  float64
 6   dang_gia_tien         16270 non-null  float64
 7   co_so_vat_chat        16270 non-null  float64
 8   do_sach_se            16270 non-null  float64
 9   bep                   16270 non-null  int64  
 10  bua_sang              16270 non-null  int64  
 11  san_gon               16270 non-null  int64  
 12  may_giat              16270 non-null  int64  
 13  dua_don_san_bay       16270 non-null  int64  
 14  phong_tap             16270 non-null  int64  
 15  don_phong_hang_ngay

In [50]:
# Chọn các tiện nghi làm feature
features = [
    'bep', 'bua_sang', 'san_gon', 
    'may_giat', 'dua_don_san_bay', 'phong_tap', 'don_phong_hang_ngay', 
    'thu_nuoi', 'bai_do_xe', 'vi_tri', 'dich_vu', 
    'dang_gia_tien', 'co_so_vat_chat', 'do_sach_se'
]

In [51]:
X = merged_df[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [52]:
# Người dùng nhập danh sách thích / không thích
yeu_thich = ['Khách sạn Thanh Mai (Hotel Thanh Mai)', 'homestay nhà sàn', 'Nice Hotel Buôn Ma Thuột']
khong_thich = ['Cozrum Homes - Sonata Residence']

merged_df['target'] = 0
merged_df.loc[merged_df['ten'].isin(yeu_thich), 'target'] = 1
merged_df.loc[merged_df['ten'].isin(khong_thich), 'target'] = -1

In [53]:
# Gom cụm khách sạn theo tiện nghi (KMeans)
kmeans = KMeans(n_clusters=4, random_state=42)
merged_df['cluster'] = kmeans.fit_predict(X_scaled)

In [54]:
# Đánh giá cụm nào giống "gu thích" hay "không thích"
def danh_gia_cum(cum_df):
    if (cum_df['target'] == 1).sum() > (cum_df['target'] == -1).sum():
        return 'GẦN GU THÍCH'
    elif (cum_df['target'] == -1).sum() > (cum_df['target'] == 1).sum():
        return 'KHÔNG HỢP GU'
    else:
        return 'TRUNG LẬP'

merged_df['cum_danh_gia'] = merged_df.groupby('cluster')['target'].transform(lambda x: danh_gia_cum(merged_df.loc[x.index]))

In [55]:
# Tạo phiên bản "khuếch đại gu" để huấn luyện Ridge
merged_df['target_khuech_dai'] = merged_df['target']
merged_df.loc[merged_df['cum_danh_gia'] == 'GẦN GU THÍCH', 'target_khuech_dai'] = 1
merged_df.loc[merged_df['cum_danh_gia'] == 'KHÔNG HỢP GU', 'target_khuech_dai'] = -1

In [56]:
# Huấn luyện mô hình Ridge để học gu từ các tiện nghi
model = Ridge(alpha=1.0)
model.fit(X_scaled, merged_df['target_khuech_dai'])

Ridge()

In [57]:
# Phân tích trọng số các tiện nghi theo gu người dùng 
feature_weights = pd.Series(model.coef_, index=features).sort_values(ascending=False)
print("🔍 Trọng số các tiện nghi phản ánh gu khách hàng:")
print(feature_weights)

🔍 Trọng số các tiện nghi phản ánh gu khách hàng:
vi_tri                 0.158089
don_phong_hang_ngay    0.128376
dang_gia_tien          0.065623
bai_do_xe              0.044774
dich_vu                0.022177
thu_nuoi               0.011711
co_so_vat_chat        -0.042207
dua_don_san_bay       -0.083780
bep                   -0.097679
may_giat              -0.100427
bua_sang              -0.122366
do_sach_se            -0.147363
phong_tap             -0.227934
san_gon               -0.294155
dtype: float64


In [58]:
# Dự đoán mức độ phù hợp của từng khách sạn
merged_df['gia_tri_thuc_ridge'] = model.predict(X_scaled)
scaler = MinMaxScaler(feature_range=(0, 10))
merged_df['gia_tri_thuc_ridge_scaled'] = scaler.fit_transform(merged_df[['gia_tri_thuc_ridge']])

merged_df['goi_y'] = np.tanh(merged_df['gia_tri_thuc_ridge'])  # Chuẩn hóa nhẹ về [-1, 1]
merged_df['gia_tri_thuc'] = 0.5 * merged_df['gia_tri_thuc_ridge'] + 0.5 * merged_df['gia_tri_thuc_netflix']
scaler_final = MinMaxScaler(feature_range=(0, 10))
merged_df['gia_tri_thuc'] = scaler_final.fit_transform(merged_df[['gia_tri_thuc']]).round(2)

In [59]:
# In danh sách khách sạn KHÁC (không trùng tên)
khach_san_khac = (
    merged_df[merged_df['target'] == 0]  # Không nằm trong danh sách thích/không thích
    [['ten', 'goi_y', 'cum_danh_gia']]
    .drop_duplicates(subset='ten')  # Loại bỏ khách sạn trùng tên
    .sort_values(by='goi_y', ascending=False)
)

In [60]:
print("📋 Danh sách KHÁCH SẠN KHÁC (không trùng tên):")
print(khach_san_khac.head(10))  # In top 10 gợi ý khác nhau

📋 Danh sách KHÁCH SẠN KHÁC (không trùng tên):
                                                     ten     goi_y  \
4697             Bin Bin Hotel 14 - Near Vstar School D7  0.872456   
4865                                Ngọc Hương Hotel HCM  0.861165   
2694                                          Tago Amber  0.850823   
2224                                  Khách Sạn Thy Thảo  0.842772   
12037                              Venus Hotel & Massage  0.832716   
44                     Van Long Lakeside View Guesthouse  0.830340   
2234   Khách Sạn Sao Nam - Phố Đi Bộ Bùi Viện (Sao Na...  0.822098   
7410                              OYO 746 Ben Thanh Dorm  0.820718   
2192                        Collection O Madam Cuc Hotel  0.817332   
597                                     Sao Khuê Hotel 1  0.814949   

       cum_danh_gia  
4697   GẦN GU THÍCH  
4865      TRUNG LẬP  
2694   GẦN GU THÍCH  
2224   GẦN GU THÍCH  
12037  GẦN GU THÍCH  
44     GẦN GU THÍCH  
2234   GẦN GU THÍCH  
7410   

## Xóa những cột không cần thiết

In [61]:
columns_to_drop = ['target', 'target_khuech_dai', 'cum_danh_gia', 'cluster', 'gia_tri_thuc_ridge', 'gia_tri_thuc_ridge_scaled', 'gia_tri_thuc_netflix', 'goi_y']
merged_df.drop(columns=columns_to_drop, inplace=True)

In [62]:
# Lưu kết quả vào file CSV
merged_df.to_csv('Gold_Data.csv', index=False)
print("✅ Đã lưu kết quả vào Gold_Data.csv")

✅ Đã lưu kết quả vào Gold_Data.csv
